# 📈 The `property` splitter family: label-shift and distributional stress tests

Welcome! The `property` family holds out on a **continuous molecular or label property** rather than on chemical structure: a descriptor value, a label range, a full label distribution, or a train/test-discriminability signal. These splitters answer *"does the model generalize to a different part of property/label space?"* rather than *"does it generalize to a different scaffold?"*

**Contents**
1. [📐 `PropertySplitter`](#1) — split along a molecular descriptor
2. [📊 `LabelExtrapolationSplitter`](#2) — extrapolate along `y`
3. [🎯 `StratifiedDistributionSplitter`](#3) — match the full label distribution
4. [🚚 `MOODSplitter`](#4) — pick the split most representative of deployment
5. [🕵️ `AdversarialSplitter`](#5) — audit or construct covariate shift

In [1]:
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

import numpy as np

from chemsplit import datasets
from chemsplit.splitters.baseline import RandomSplitter
from chemsplit.splitters.property_ import (
    AdversarialSplitter,
    LabelExtrapolationSplitter,
    MOODSplitter,
    PropertySplitter,
    StratifiedDistributionSplitter,
)

fx = datasets.make_scaffold_families(n_scaffolds=6, per_scaffold=15, seed=0)
smiles = fx.smiles
print(f"{len(smiles)} molecules across {len(set(fx.groups_true.tolist()))} scaffold families")

def summarize(name, result):
    print(f"{name}: train={result.train.size} valid={result.valid.size} "
          f"test={result.test.size} discard={result.discard.size} (n={result.n_records})")

90 molecules across 6 scaffold families


<a id="1"></a>
## 1. 📐 `PropertySplitter`

A single sorted cut on an RDKit descriptor (or a caller-supplied callable/array of precomputed values): the highest, lowest, both tails, or the central band become `test`. The validation band, when requested, is drawn adjacent to `test` on the train side so early stopping shares the same extrapolation direction as the test evaluation.

| Parameter | Meaning |
|---|---|
| `property` | RDKit descriptor name (or callable), e.g. `"MolWt"` |
| `direction` | which end(s) of the sorted values become `test`: `"high_test"`, `"low_test"`, `"extremes_test"`, `"middle_test"` |
| `train_size` / `test_size` | target fractions |

> 💡 **Advantages**
> - Directly models real extrapolations: fragment-to-lead growth (MW), applying a small-molecule model to peptides or PROTACs (size), and solubility/permeability range shifts (logP, TPSA).
> - Fully deterministic and easy to explain — `train_range` vs. `test_range` and `overlap` state exactly what was asked of the model.
> - No featurization, clustering, or seed needed — the cheapest hard split available.

> ⚠️ **Pitfalls**
> - Molecular weight correlates with almost everything, including assay artefacts, promiscuity, and the era a series was made — a performance drop may be confounded rather than caused by the property shift.
> - Because it's a single sorted cut, the test set is chemically homogeneous with strongly correlated errors, so the effective sample size is well below `n_test`.
> - **Not** a leakage-control split — a test molecule can be a close analogue of a training molecule that just happens to sit below the cut. Check `metadata["overlap"]` and `audit.nn_similarity_profile`.
> - `direction="middle_test"` is an interpolation test despite living in this family — don't report it as extrapolation.

In [2]:
sp = PropertySplitter(property="MolWt", direction="high_test", train_size=0.7, test_size=0.3)
result = sp.split_result(smiles)[0]
summarize("PropertySplitter", result)
print("train MolWt range:", result.metadata["train_range"], " test MolWt range:", result.metadata["test_range"])

PropertySplitter: train=63 valid=0 test=27 discard=0 (n=90)
train MolWt range: [78.11399999999999, 157.01]  test MolWt range: [157.21599999999995, 208.05799999999996]


<a id="2"></a>
## 2. 📊 `LabelExtrapolationSplitter`

Train on one part of the label range, test on another — an extrapolation stress-test on `y` itself, not on a molecular descriptor.

| Parameter | Meaning |
|---|---|
| `direction` | `"high_test"`, `"low_test"`, or `"extremes_test"` |
| `buffer` | gap between train and test in label units (float) or records (int); records inside go to `discard` |

> 💡 **Advantages**
> - The most honest test of whether a model can rank compounds *better than its training data* — the real requirement for generative design and prioritising untested potency ranges.
> - `buffer` makes the extrapolation gap explicit and tunable, so performance can be reported as a function of gap width.
> - Needs no chemistry at all, so it works for any modality.

> ⚠️ **Pitfalls**
> - Brutal by construction — most regression models regress to their training mean and under-predict the held-out extreme, so a flat prediction can post a respectable RMSE with zero rank correlation. **Always report a ranking metric (Spearman, top-k enrichment) alongside RMSE/R²**.
> - Selecting the split with the labels makes it label-aware by construction: label noise at the extreme directly shapes the test population.
> - Extreme labels concentrate measurement artefacts, censored values, and transcription errors — exactly where data quality is worst.
> - On binary labels this becomes a class holdout where the model never sees a positive example — a different, usually pointless, experiment.

In [3]:
n = len(smiles)
y = np.linspace(0, 10, n)
sp = LabelExtrapolationSplitter(direction="high_test", buffer=5, train_size=0.6, test_size=0.3)
result = sp.split_result(smiles, y)[0]
summarize("LabelExtrapolationSplitter", result)
print("buffer_records (discarded gap):", result.metadata["buffer_records"])

LabelExtrapolationSplitter: train=49 valid=9 test=27 discard=5 (n=90)
buffer_records (discarded gap): 5


<a id="3"></a>
## 3. 🎯 `StratifiedDistributionSplitter`

Match the full label *distribution* — not just the mean or class balance — between train and test, optionally optimizing a Kolmogorov-Smirnov statistic.

| Parameter | Meaning |
|---|---|
| `match` | `"histogram"` (per-bin apportionment, deterministic), `"moments"` (seeded hill-climb on mean/std/skew), or `"ks"` (restart until two-sample KS ≤ `max_ks`) |
| `n_bins` | number of bins used for matching |

> 💡 **Advantages**
> - Cuts metric variance on small regression datasets more effectively than class-level stratification, since it matches shape rather than just balance.
> - Reports `ks_statistic`, so "the partitions share a label distribution" is evidenced, not assumed.
> - Deterministic in its default `histogram` mode.

> ⚠️ **Pitfalls**
> - Changes nothing about chemical leakage — it's `stratified_random` with more bins, and calling it a "distribution-matched split" invites more rigour than it actually has.
> - Matching the label distribution makes the test set *easier* by construction, removing exactly the label shift a real prospective test would contain.
> - `match="ks"` can loop to the restart limit on tied or censored labels.

In [4]:
rng = np.random.default_rng(0)
y = rng.normal(size=n)
sp = StratifiedDistributionSplitter(match="ks", n_bins=10, max_ks=0.3, max_restarts=5, train_size=0.7, test_size=0.3, random_state=0)
result = sp.split_result(smiles, y)[0]
summarize("StratifiedDistributionSplitter", result)
print("KS statistic between train/test label distributions:", round(result.metadata["ks_statistic"], 4))

StratifiedDistributionSplitter: train=60 valid=0 test=30 discard=0 (n=90)
KS statistic between train/test label distributions: 0.1


<a id="4"></a>
## 4. 🚚 `MOODSplitter`

Select, among several candidate splitters, the one whose train→test distance distribution best matches the train→**deployment** distance distribution (MOOD: "Massive Out-Of-Distribution shift" splitter) — i.e. picks the split that is *most representative* of production, rather than the "hardest" one.

| Parameter | Meaning |
|---|---|
| `candidates` | already-instantiated candidate splitters (string ids are rejected) |
| `deployment_set` | the library you actually intend to screen — required |
| `distance_stat` / `discrepancy` | how distances and distributional mismatch are measured |

> 💡 **Advantages**
> - Reframes "which split is hardest?" as "which split is *representative* of my deployment?" — the only version of the question with a defensible answer.
> - Produces an auditable table of candidate scores, so the choice is evidence rather than taste.
> - Automatically falls back to a *random* split when that's genuinely appropriate.

> ⚠️ **Pitfalls**
> - Requires the deployment library up front; without it the method is undefined, and a guessed deployment set silently decides the answer.
> - The selected split is chosen using a statistic computed from the data, so the reported score is mildly optimistic in a model-selection sense.
> - Running five candidate splitters costs five splits, which is expensive on `O(n²)` candidates.

In [5]:
deploy = smiles[:10]
candidates = (
    RandomSplitter(train_size=0.7, test_size=0.3, random_state=0),
    RandomSplitter(train_size=0.7, test_size=0.3, random_state=1, shuffle=False),
)
sp = MOODSplitter(candidates=candidates, deployment_set=deploy, train_size=0.7, test_size=0.3, random_state=0)
result = sp.split_result(smiles)[0]
summarize("MOODSplitter", result)
print("candidate_scores:", result.metadata["candidate_scores"])
print("selected:", result.metadata["selected"])

MOODSplitter: train=63 valid=0 test=27 discard=0 (n=90)
candidate_scores: [['random', 0.47325849146754656], ['random', 0.7023464364034159]]
selected: random


`featurizer`/`metric` (inherited from `SimilarityParamsMixin`, default `"ecfp4"`/`"tanimoto"`) define what "distance to deployment" even means here — MOOD's whole selection criterion is computed in that feature space, so switching to a continuous descriptor set changes which candidate split gets picked.

In [6]:
sp_physchem = MOODSplitter(
    candidates=candidates, deployment_set=deploy, featurizer="physchem", metric="cosine",
    train_size=0.7, test_size=0.3, random_state=0,
)
result_physchem = sp_physchem.split_result(smiles)[0]
summarize("MOODSplitter (physchem/cosine)", result_physchem)
print("selected:", result_physchem.metadata["selected"], "| candidate_scores:", result_physchem.metadata["candidate_scores"])

MOODSplitter (physchem/cosine): train=63 valid=0 test=27 discard=0 (n=90)
selected: random | candidate_scores: [['random', 0.0012996488586833927], ['random', 0.023308679468452026]]


<a id="5"></a>
## 5. 🕵️ `AdversarialSplitter`

Train a train-vs-test discriminator on an existing split (`mode="audit"`, reporting how separable train/test are) or search for a split with a *target* discriminability (`mode="construct"`).

| Parameter | Meaning |
|---|---|
| `mode` | `"audit"` (score an existing split) or `"construct"` (search for `target_auc`) |
| `target_auc` | desired train/test discriminability in construct mode |
| `classifier` | `"logreg"` or `"gbdt"` discriminator |

> 💡 **Advantages**
> - `mode="audit"` is the cheapest possible check that a split is what it claims — one comparable number across splitters, datasets, and papers.
> - `mode="construct"` lets you dial covariate shift to a chosen level and measure degradation as a function of shift.
> - `top_discriminative_features` names the bits or descriptors separating the partitions, often revealing an unintended confound.

> ⚠️ **Pitfalls**
> - A constructed split is optimised against a specific discriminator on specific features — a different model may see no shift at all.
> - Pushing AUC up tends to surface *trivial* separations first (molecular size, a common substructure) rather than chemically interesting shift.
> - Convergence isn't guaranteed and is only reported, not enforced.

The `AdversarialSplitter(mode="construct")` cell below may emit a benign scikit-learn `FutureWarning` about the logistic-regression `penalty` parameter from its internal discriminator — harmless, not an error.

In [7]:
sp_audit = AdversarialSplitter(mode="audit", train_size=0.7, test_size=0.3, random_state=0)
r_audit = sp_audit.split_result(smiles)[0]
summarize("AdversarialSplitter (audit)", r_audit)
print("audit final_auc:", round(r_audit.metadata["final_auc"], 4))

sp_construct = AdversarialSplitter(mode="construct", target_auc=0.6, max_iter=3, train_size=0.7, test_size=0.3, random_state=0)
r_construct = sp_construct.split_result(smiles)[0]
summarize("AdversarialSplitter (construct)", r_construct)
print("construct final_auc:", round(r_construct.metadata["final_auc"], 4), " iterations:", r_construct.metadata["iterations"])

AdversarialSplitter (audit): train=63 valid=0 test=27 discard=0 (n=90)
audit final_auc: 0.4955


AdversarialSplitter (construct): train=63 valid=0 test=27 discard=0 (n=90)
construct final_auc: 0.5903  iterations: 2


`classifier="gbdt"` (gradient-boosted trees) swaps in a nonlinear discriminator instead of the default `"logreg"` — it can pick up interactions a linear model misses, so the audited AUC (or the covariate shift `construct` mode converges to) can differ meaningfully between the two.

In [8]:
sp_audit_gbdt = AdversarialSplitter(mode="audit", classifier="gbdt", train_size=0.7, test_size=0.3, random_state=0)
r_audit_gbdt = sp_audit_gbdt.split_result(smiles)[0]
summarize("AdversarialSplitter (audit, gbdt)", r_audit_gbdt)
print("logreg final_auc:", round(r_audit.metadata["final_auc"], 4),
      "| gbdt final_auc:", round(r_audit_gbdt.metadata["final_auc"], 4))

AdversarialSplitter (audit, gbdt): train=63 valid=0 test=27 discard=0 (n=90)
logreg final_auc: 0.4955 | gbdt final_auc: 0.3144


---
That covers all 5 `property`-family splitters. Pair them with `chemsplit.audit` (nearest-neighbour similarity, adversarial validation) to confirm a split's stress is the *intended* kind before reporting results on it.